# 01 · Retain and verify NetEaseCrowd
Real proxy dataset; no synthetic outcomes or substituted assignments.

In [1]:
from pathlib import Path
import sys, json, inspect
ROOT=Path.cwd().resolve()
if ROOT.name=='notebooks': ROOT=ROOT.parent
if str(ROOT.parent) not in sys.path: sys.path.insert(0,str(ROOT.parent))
from ml.src.preprocessing import load_data, read_json, save_json
from ml.src.evaluation import evaluate, feature_groups
from ml.src.notebook_helpers import table, metric_row, chart
from IPython.display import display, Code
import matplotlib.pyplot as plt
%matplotlib inline


Source: https://github.com/fuxiAIlab/NetEaseCrowd-Dataset (CC-BY-SA 4.0). All three original partitions remain available, containing 1,203,264 raw rows. We retain the existing deterministic task sample and temporal splits to reproduce the prior experiment. This is the full usable **existing experiment**, not the full public dataset. Worker/task IDs stay strings. Earliest duplicate responses were retained, conflicting task labels are rejected, and boundary-crossing tasks are purged.

In [2]:
from ml.src.preprocessing import verify_source_rows
groups,splits=load_data()
verified=verify_source_rows(groups)
print('Verified original CSV source rows:',verified)
table([{'Split':s,'Tasks':len(ids),'Observations':sum(len(groups[t]) for t in ids)} for s,ids in splits.items()])
print('Workers:',len({r['agent_id'] for g in groups.values() for r in g}))
print('Capabilities:',sorted({r['capability'] for g in groups.values() for r in g}))
table([r for g in list(groups.values())[:2] for r in g][:6])

Verified original CSV source rows: 121201


Split,Tasks,Observations
train,12473,85647
validation,2018,12316
test,2699,17137
purged,797,6101


Workers: 1653
Capabilities: ['50', '53', '56', '69']


task_id,agent_id,capability,taskset_id,answer,truth,timestamp,success,source_file,source_line,split
1012658485080359232,150,69,6980,1,0,1661919314848,0,NetEaseCrowd_part_1.csv,215,train
1012658485080359232,314,69,6980,0,0,1661972566168,1,NetEaseCrowd_part_1.csv,216,train
1012658485080359232,372,69,6980,0,0,1661916487229,1,NetEaseCrowd_part_1.csv,217,train
1012658485080359232,375,69,6980,0,0,1661931854649,1,NetEaseCrowd_part_1.csv,218,train
1012658485080359232,440,69,6980,0,0,1661989589798,1,NetEaseCrowd_part_1.csv,219,train
1012658485080359232,456,69,6980,0,0,1661848187751,1,NetEaseCrowd_part_1.csv,220,train


## Leakage prevention
All feature histories and graph edges are constructed from training rows only. Test membership identifies the observed candidate pool; test answers/truth are labels only. Synthetic display fields never enter benchmark features. The saved legacy model is checked against exact training counts before reuse.

In [3]:
from ml.src.hybrid import prepare_artifacts
stats=prepare_artifacts()
table([stats])
profiles=read_json('models/netease/display_profiles.json')
table([{k:p[k] for k in ('source_worker_id','agent_id','agent_name','historical_task_count','historical_success_rate','location')} for p in list(profiles.values())[:6]])

nodes,edges,node_types,edge_types,training_observations,graph_input
13985,100154,"{'Worker': 1508, 'Task': 12473, 'Capability': 4}","{'COMPLETED': 85647, 'HAS_CAPABILITY': 2034, 'REQUIRES_CAPABILITY': 12473}",85647,training split only


source_worker_id,agent_id,agent_name,historical_task_count,historical_success_rate,location
10000,AGENT_0001,Amina Hassan,13,0.7692307692307693,Nairobi
10001,AGENT_0002,Brian Hassan,0,None,Mombasa
10013,AGENT_0003,Faith Hassan,2,1.0,Kisumu
10014,AGENT_0004,Kevin Hassan,246,0.6504065040650406,Nakuru
10016,AGENT_0005,Sharon Hassan,4,1.0,Eldoret
10018,AGENT_0006,David Hassan,0,None,Nairobi


Agent names are synthetic display identities used for demonstration and privacy. The underlying historical interactions originate from the NetEaseCrowd dataset. Kenyan locations and initial workload/availability are synthetic demonstration attributes; historical counts/performance come only from training. Each profile stores field provenance.